<a href="https://colab.research.google.com/github/hyperdbio/Android-inject/blob/master/ex02_YOLO_%EA%B0%9D%EC%B2%B4%ED%83%90%EC%A7%80(%EB%8F%99%EB%AC%BC%EB%8D%B0%EC%9D%B4%ED%84%B0).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 전처리 cpu
# 학습 gpu
# 드라이브 연동
# 경로 이동 yolo_study

%cd /content/drive/MyDrive/YOLO_study

/content/drive/MyDrive/YOLO_study


## 목표
- 동물객체를 탐지하는 모델을 만들어보자
 (Object Detection)
- 객체 탐지에 필요한 기본 개념을 익혀보자
    - YOLO 모델 데이터 준비
    - YOLO 모델 선택 및 학습
    - 모델 성능 확인(평가지표): IoU, mAP
    

In [ ]:
%pwd

'/content'

### 데이터 준비하기
 - Animals-10 캐글 데이터셋 링크:

 - 데이터셋: 네이버 이미지 탭 출처 친칠라 이미지 -> roboflow 라벨링
- 코드 형식으로 다운로드 받는 작업
- data.yaml 경로를 설정(상대경로 -> 절대경로)




 https://www.kaggle.com/datasets/alessiocorrado99/animals10


In [ ]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 110.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="PZ8mvrVzaMkYewZYoGak")
project = rf.workspace("ysdb").project("chin_detect-hyyom")
version = project.version(2)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to chin_detect-2 in yolov11:: 100%|██████████| 219/219 [00:01<00:00, 110.85it/s]


In [ ]:
%pwd

'/content/drive/MyDrive/YOLO_study'

In [ ]:
# 0 0.48984375 0.48203125 0.9796875 0.9640625
# class x_center y_center width height

# chin - 0

- **YOLO 라벨 포맷의 기본 구조**  
  - 하나의 이미지에 대한 라벨 정보를 `*.txt` 파일에 기록  
  - 이미지에 객체가 하나도 없다면 해당 `*.txt` 파일 자체가 필요 없음  
  - 한 줄에 하나의 객체 정보를 기록하며, 포맷은 아래와 같음  
    \[ `class x_center y_center width height` \]  
  - `class` 번호는 0부터 시작(0-indexed)

- **좌표값은 정규화(normalized)되어야 함**  
  - 가로(너비)와 세로(높이)를 0~1 사이로 변환  
  - (x_center, width)는 이미지 실제 너비(픽셀 수)로 나누어 0~1 범위로 만듦  
  - (y_center, height)는 이미지 실제 높이(픽셀 수)로 나누어 0~1 범위로 만듦  
  - 예: 이미지의 폭이 1000px이고, 어떤 박스의 중심 x 좌표가 500px이라면,  
    x_center = 500 / 1000 = 0.5

- **(x_center, y_center)는 박스 중심점의 좌표**  
  - 일반적인 바운딩박스(왼쪽 상단 x, 왼쪽 상단 y) 방식과 달리,  
    YOLO는 “박스 중심점” + “박스 너비/높이” 로 표현  
  - 따라서, 좌표 변환 시 박스 중심점을 기준으로 잡아야 함

- **width, height 역시 픽셀 기준이 아닌 정규화된 값**  
  - 박스의 실제 너비(픽셀)를 전체 이미지 너비로 나누고,  
    실제 높이(픽셀)를 전체 이미지 높이로 나누어 0~1 사이로 변환  
  - 예: 이미지 높이가 800px, 박스 높이가 400px이라면  
    height = 400 / 800 = 0.5

- **결과 예시**  
  - 클래스가 0번이고,  
  - 이미지의 폭과 높이가 각각 1000px, 800px인 상황에서,  
  - 객체 중심이 (500px, 300px), 박스 크기가 (200px × 100px)라고 할 때  
    - x_center = 500 / 1000 = 0.5  
    - y_center = 300 / 800 = 0.375  
    - width = 200 / 1000 = 0.2  
    - height = 100 / 800 = 0.125  
    - 최종 라벨: `0 0.5 0.375 0.2 0.125`

- **요약**  
  1. 이미지를 기준으로 박스의 중심(x_center, y_center)을 구함  
  2. 박스의 실제 폭(width), 높이(height)를 구함  
  3. 각각 이미지 폭/높이로 나누어 0~1 사이 값으로 만듦(정규화)  
  4. 라벨 파일(`*.txt`)에 `[클래스 x_center y_center width height]` 형태로 기록  

In [ ]:
# data.yaml
# train, test 경로를 설정하는 부분
# ===========================================================

# train: /content/drive/MyDrive/YOLO_study/chin_detect-2/train
# val: ../valid/images
# test: /content/drive/MyDrive/YOLO_study/chin_detect-2/test

# nc: 1
# names: ['chin']

# roboflow:
#   workspace: ysdb
#   project: chin_detect-hyyom
#   version: 2
#   license: CC BY 4.0
#   url: https://universe.roboflow.com/ysdb/chin_detect-hyyom/dataset/2

### YOLOv11 Detection 사전학습 모델 학습시키기


In [ ]:
!pip -q install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.9 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo11n.pt") # load a
model

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_

In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo11n.pt")  # load a pretrained model (recommended for training)

# Train the model
results = model.train(data="coco8.yaml", epochs=100, imgsz=640)

Ultralytics 8.3.203 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train8, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=

In [ ]:
# Train the model
results = model.train(data="coco8.yaml", epochs=100, imgsz=640)

Ultralytics 8.3.203 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train82, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained

In [ ]:
# mAP50 : 전반적인 예측 탐지값에 대한 평균 점수를 확인한 것
# mAP50-95 : 예측을 잘 해낸 것(50% 이상정도 탐지를 잘 한것)에 대해 평균 점수를 확인한 것
# mAP50 보다 mAP50-90의 평가지표 조금 더 확실한 모델의 성능을 파악할 수 있음

# 현재: 우리 모델은 생각보다 친칠라 탐지에 약한 상태

In [ ]:
# 모델 예측
# model 변수에 담긴 모델 활용
# best.pt 불러와서 model 예측 활용
# /content/drive/MyDrive/YOLO_study/runs/detect/train4
best_model = ('./runs/detect/train4/weights/best.pt')


# from ultralytics import YOLO

# # best.pt 경로 지정
# model = YOLO('/content/drive/MyDrive/YOLO_study/runs/detect/train4/weights/best.pt')

# # 이미지 예측 실행
# results = model('/content/drive/MyDrive/YOLO_study/test_image.jpg', save=True)


In [ ]:
results = model('/content/drive/MyDrive/YOLO_study/test_image.jpg', save=True)

FileNotFoundError: /content/drive/MyDrive/YOLO_study/test_image.jpg does not exist

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/YOLO_study/runs/detect/train4/weights/best.pt")

In [ ]:
model = YOLO("/content/drive/MyDrive/YOLO_study/runs/classify/train/weights")

# 이미지 추론
results = model("test.jpg")

# 결과 출력
for r in results:
    print(r.boxes.cls)
    print(r.boxes.conf)

TypeError: model='/content/drive/MyDrive/YOLO_study/runs/classify/train/weights' is not a supported model format. Ultralytics supports: ('PyTorch', 'TorchScript', 'ONNX', 'OpenVINO', 'TensorRT', 'CoreML', 'TensorFlow SavedModel', 'TensorFlow GraphDef', 'TensorFlow Lite', 'TensorFlow Edge TPU', 'TensorFlow.js', 'PaddlePaddle', 'MNN', 'NCNN', 'IMX', 'RKNN')
See https://docs.ultralytics.com/modes/predict for help.